In [26]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI
import IPython

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI. 
The env file should look like this :

```
OPENAI_API_KEY=xxx
GOOGLE_API_KEY=xxxx
ANTHROPIC_API_KEY=xxxx
DEEPSEEK_API_KEY=xxxx
HF_TOKEN=xxx
AZURE_OPENAI_API_KEY=xxx
```

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
openai = OpenAI()

# If this doesn't work, try Kernel menu >> Restart Kernel and Clear Outputs Of All Cells, then run the cells from the top of this notebook down.
# If it STILL doesn't work (horrors!) then please see the Troubleshooting notebook in this folder for full instructions

## And now let's build useful messages for GPT-4o-mini, using a function

# OpenAI Parameters

## 1. `model`
Specifies which model to use (e.g., `gpt-4`, `gpt-3.5-turbo`).  
- Different models have different capabilities, speed, and cost.  

## 2. `temperature`
Controls randomness in the output.  
- Range: `0.0 – 2.0`  
- Lower values → more deterministic, focused answers.  
- Higher values → more creative, varied responses.  

## 3. `max_tokens`
The maximum number of tokens (words + pieces of words) the model can generate.  
- Helps limit output length.  

## 4. `top_p`
Nucleus sampling parameter.  
- Range: `0.0 – 1.0`  
- The model considers only the most probable tokens that together have probability `p`.  
- Alternative to `temperature`.  

## 5. `frequency_penalty`
Controls how much to reduce the chance of repeating the same line/phrase.  
- Range: `-2.0 – 2.0`  
- Higher values → discourage repetition.  

## 6. `presence_penalty`
Controls how much to encourage the model to talk about new topics.  
- Range: `-2.0 – 2.0`  
- Higher values → increase diversity by discouraging sticking to the same topics.  


In [30]:
def set_open_params(
    model="gpt-3.5-turbo",
    temperature=0.7,
    max_tokens=256,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
):
    """ set openai parameters"""

    openai_params = {}    

    openai_params['model'] = model
    openai_params['temperature'] = temperature
    openai_params['max_tokens'] = max_tokens
    openai_params['top_p'] = top_p
    openai_params['frequency_penalty'] = frequency_penalty
    openai_params['presence_penalty'] = presence_penalty
    return openai_params

def get_completion(params, messages):
    """ GET completion from openai api"""

    response = openai.chat.completions.create(
        model = params['model'],
        messages = messages,
        temperature = params['temperature'],
        max_tokens = params['max_tokens'],
        top_p = params['top_p'],
        frequency_penalty = params['frequency_penalty'],
        presence_penalty = params['presence_penalty'],
    )
    return response

# Prompting Tasks in LLMs / GenAI

## 1. Question Answering (QA)
- **Definition:** Designing prompts so the model gives direct, fact-based answers to queries.  
- **Use Cases:** Chatbots, knowledge-base assistants, RAG systems.  
- **Example Prompt:**  
  > "Answer the following question based only on the provided context: [context here]"

---

## 2. Text Classification
- **Definition:** Using prompts to categorize text into labels like sentiment, intent, or topic.  
- **Use Cases:** Spam detection, sentiment analysis, ticket routing.  
- **Example Prompt:**  
  > "Classify the following text as Positive, Negative, or Neutral: 'This movie was amazing!'"

---

## 3. Role Playing
- **Definition:** Prompting the LLM to simulate a persona or role and respond accordingly.  
- **Use Cases:** Customer support simulations, interview practice, AI agents.  
- **Example Prompt:**  
  > "You are a financial advisor. Provide advice as if you are guiding a client with moderate risk appetite."

---

## 4. Code Generation
- **Definition:** Using prompts to generate executable code, scripts, or tests.  
- **Use Cases:** GitHub Copilot, Cursor IDE, boilerplate generation, automation scripts.  
- **Example Prompt:**  
  > "Write a Python function to calculate Fibonacci numbers using recursion."


### 1.1 Text Summarization

In [28]:
params = set_open_params(temperature=0.7)
prompt = """Antibiotics are a type of medication used to treat bacterial infections. They work by either killing the bacteria or preventing them from reproducing, allowing the body's immune system to fight off the infection. Antibiotics are usually taken orally in the form of pills, capsules, or liquid solutions, or sometimes administered intravenously. They are not effective against viral infections, and using them inappropriately can lead to antibiotic resistance. 

Explain the above in one sentence:"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)

Antibiotics are medications used to treat bacterial infections by either killing the bacteria or preventing their reproduction, but they are not effective against viral infections and should be used appropriately to avoid antibiotic resistance.

Exercise: Instruct the model to explain the paragraph in one sentence like "I am 5". Do you see any differences?

### 1.2 Question Answering

In [19]:
prompt = """Answer the question based on the context below. Keep the answer short and concise. Respond "Unsure about answer" if not sure about the answer.

Context: Teplizumab traces its roots to a New Jersey drug company called Ortho Pharmaceutical. There, scientists generated an early version of the antibody, dubbed OKT3. Originally sourced from mice, the molecule was able to bind to the surface of T cells and limit their cell-killing potential. In 1986, it was approved to help prevent organ rejection after kidney transplants, making it the first therapeutic antibody allowed for human use.

Question: What was OKT3 originally sourced from?

Answer:"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)


Mice.

Context obtained from here: https://www.nature.com/articles/d41586-023-00400-x

Exercise: Edit prompt and get the model to respond that it isn't sure about the answer. 

### 1.3 Text Classification

In [21]:
prompt = """Classify the text into neutral, negative or positive.

Text: I think the food was okay.

Sentiment:"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)

Neutral

Exercise: Modify the prompt to instruct the model to provide an explanation to the answer selected. 

### 1.4 Role Playing

In [29]:
prompt = """The following is a conversation with an AI research assistant. The assistant tone is technical and scientific.

Human: Hello, who are you?
AI: Greeting! I am an AI research assistant. How can I help you today?
Human: Can you tell me about the creation of blackholes?
AI:"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)

Black holes are formed when a massive star reaches the end of its life cycle and collapses under its own gravity. This collapse causes the star's core to become extremely dense, forming a singularity with an intense gravitational pull. Surrounding matter is then pulled towards the singularity, creating an event horizon beyond which nothing, not even light, can escape. This is what gives black holes their characteristic "black" appearance.

Exercise: Modify the prompt to instruct the model to keep AI responses concise and short.

### 1.5 Code Generation

In [23]:
prompt = "\"\"\"\nTable departments, columns = [DepartmentId, DepartmentName]\nTable students, columns = [DepartmentId, StudentId, StudentName]\nCreate a MySQL query for all students in the Computer Science Department\n\"\"\""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)


SELECT StudentName
FROM students
WHERE DepartmentId = (SELECT DepartmentId FROM departments WHERE DepartmentName = 'Computer Science')

### 1.6 Reasoning

In [24]:
prompt = """The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1. 

Solve by breaking the problem into steps. First, identify the odd numbers, add them, and indicate whether the result is odd or even."""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)

To solve this problem, we need to follow these steps:

Step 1: Identify the odd numbers in the given group. The odd numbers in the group are 15, 5, 13, 7, and 1.

Step 2: Add the odd numbers together. 15 + 5 + 13 + 7 + 1 = 41.

Step 3: Determine whether the sum is odd or even. In this case, the sum is 41, which is an odd number.

Therefore, the sum of the odd numbers in the given group is an odd number.

Exercise: Improve the prompt to have a better structure and output format.